# 05 · The local model (control scientist)

A session whose plan comes from the platform's own model, Qwen3-4B, running on this host (no data leaves it).
Each turn takes about 1 to 2 minutes on the CPU. Exercise E49 in the platform manual follows this notebook.

In [ ]:
from medcea_agent import Client

c = Client()
c.signin()          # open the link, sign in, confirm the code; the cell waits

## 1. Choose the model, then a supervised session

`c.endpoints()` lists the models this install offers (the model picker): the local one runs on this host,
others on a server an administrator configured. Every one gets the same instructions, tools and gate; the
trace records which model answered. Leave `endpoint` out for the profile's own model (the local one).

The model plans: it estimates the campaign and runs it (small enough to need no gate), reads the results,
then proposes a PID setting. `watch` shows every turn; it ends when the agent stops.

In [ ]:
for e in c.endpoints():
    print(e["name"], "·", e["model"], "·", e["context"], "tokens ·", e["where"])

In [ ]:
endpoint = None      # or one of the names above, e.g. "gpu"
sid = c.start("compare_controllers_local", endpoint=endpoint)["sid"]
print(sid)
c.watch(sid)

## 2. Turn times

A turn's time is mostly the answer being written, about 2 tokens a second, plus the prompt tokens that are new since the last turn (about 9 a second). The model server keeps the conversation so far in its prompt cache, so each turn reads only what was added. Only the first session after the server starts reads the whole standing text and tool list (about 960 tokens, roughly 100 s more).

In [ ]:
t = c.turns(c.events(sid))
for turn, (req, resp, secs) in sorted(t.items()):
    print("turn %d: %s s" % (turn, secs))
c.info(sid)["spent"]

## 3. Re-execution

The same profile once more. The model is greedy with a fixed seed, so while it is sent the same request it
should give the same answer; once a tool result differs (an estimate is timed), the conversations part.
Same request is not quite enough: the answer can also change with what the model server already holds in its
prompt cache (24 Nov 2026: one turn-1 request gave a 120-token call from a cold cache and a runaway answer
from a warm one), so re-execute right after section 1, as here.
This takes as long as section 1 again.

In [ ]:
r = c.reexecute(sid)
r

## 4. Replay

The strict replay of section 1's session, from its record alone (no model): it must be `ok`.

In [ ]:
c.replay(sid)

## 5. The full comparison on the larger model

`compare_controllers_gpu` runs all five controllers in one campaign on the endpoint called `gpu` (when this install
has one: otherwise the start is refused) and proposes one setting with the criterion it is better on. mcp-sim, not the
model, checks the proposal against the campaign: a setting another one beats is refused, the refusal names the better
one, and an accepted proposal carries the server's comparison (`check`). Exercise E58 follows this section.

In [ ]:
sid_gpu = c.start("compare_controllers_gpu")["sid"]
c.watch(sid_gpu)
# every proposal the model made, refused or accepted, as mcp-sim answered it
for e in c.events(sid_gpu):
    p = e.get("payload") or {}
    if e["event"] == "tool_result" and p.get("tool") == "propose.parameter_set":
        print("refused:" if p.get("error") else "accepted:", p.get("result"))